# Foundation Model Embeddings vs. Time-Series Baselines (`catch22` & Double Logistic Curves)
Evaluates standalone time-series baselines (`B1`–`B5`) and Foundation Model fusion (`M1`–`M6`) across Random Forest and XGBoost using Leave-One-Year-Out Cross-Validation (LOYOCV).

### 1. Imports & Directory Setup

In [ ]:
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycatch22
from scipy.optimize import curve_fit
from scipy.stats import randint, uniform
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut, RandomizedSearchCV
from tqdm import tqdm
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

# Define paths matching existing notebook structure
output_dir = Path("output")
results_dir = Path("Results")
results_dir.mkdir(parents=True, exist_ok=True)

### 2. Interactive Study Folder & Foundation Model Selection

In [ ]:
subfolders = sorted([f for f in output_dir.iterdir() if f.is_dir()])

if not subfolders:
    raise FileNotFoundError(f"No subdirectories found inside '{output_dir}'.")

if len(subfolders) == 1:
    selected_folder = subfolders[0]
    print(f"Only one folder found. Automatically selected: {selected_folder.name}.")
else:
    print("Available folders in output directory:")
    for idx, folder in enumerate(subfolders, start=1):
        print(f"[{idx}] {folder.name}")

    user_choice = input(f"\nEnter folder number (1-{len(subfolders)}): ").strip()
    if not user_choice.isdigit() or not (1 <= int(user_choice) <= len(subfolders)):
        raise ValueError(
            f"Invalid selection '{user_choice}'. Expected number between 1 and {len(subfolders)}."
        )
    selected_folder = subfolders[int(user_choice) - 1]
    print(f"\n --> Selected folder: {selected_folder.name}")

candidate_files = sorted(selected_folder.glob("curated_*_embeddings.csv"))
if not candidate_files:
    raise FileNotFoundError(
        f"No embedding CSV files found inside '{selected_folder.name}'."
    )

name_mapping = {
    "alphaearth": "AlphaEarth (64-d)",
    "tessera": "TESSERA (128-d)",
    "presto": "PRESTO (128-d)",
    "olmo": "OlmoEarth (128-d)",
}

if len(candidate_files) == 1:
    chosen_file = candidate_files[0]
    print(f"Only one embedding file found: {chosen_file.name}")
else:
    print("Select which embedding representation to model:")
    for idx, file_path in enumerate(candidate_files, start=1):
        emb_tag = next(
            (
                value
                for key, value in name_mapping.items()
                if key in file_path.name.lower()
            ),
            "Unknown",
        )
        print(f"[{idx}] {file_path.name}  --> [{emb_tag}]")

    user_file_idx = input(
        f"\nEnter file number (1-{len(candidate_files)}): "
    ).strip()
    if not user_file_idx.isdigit() or not (
        1 <= int(user_file_idx) <= len(candidate_files)
    ):
        raise ValueError(f"Invalid selection '{user_file_idx}'.")
    chosen_file = candidate_files[int(user_file_idx) - 1]

print(f"\n--> Selected modeling file: {chosen_file.name}")

file_name = chosen_file.name.lower()
emb_name = next((key for key in name_mapping if key in file_name), "unknown")

folder_parts = selected_folder.name.split("_")
model = emb_name.upper()
country = folder_parts[0].capitalize()
crop = folder_parts[1].upper()
bbch = f"BBCH {folder_parts[2].replace('bbch', '')}"
start_year = folder_parts[3][:4]
end_year = folder_parts[4][:4]

final_name = f"{country}, {crop}, {bbch}, Period: {start_year}-{end_year}"
print(f"Study Metadata: {final_name} | Active FM: {model}")

### 3. Load Dataset, Clean Rows & Detect Daily Channels

In [ ]:
df_raw = pd.read_csv(chosen_file, sep=";")
df_raw["Year"] = df_raw["Year"].astype(int)

# Sort daily columns numerically (doy_1 to doy_365) so time series order is exact
agdd_cols = sorted(
    [col for col in df_raw.columns if col.startswith("AGDD_doy_")],
    key=lambda x: int(x.split("_")[-1]),
)
gdd_cols = sorted(
    [col for col in df_raw.columns if col.startswith("GDD_doy_")],
    key=lambda x: int(x.split("_")[-1]),
)
ndvi_cols = sorted(
    [col for col in df_raw.columns if col.startswith("NDVI_doy_")],
    key=lambda x: int(x.split("_")[-1]),
)

if any(col.startswith(f"{emb_name}") for col in df_raw.columns):
    embedding_cols = sorted(
        [c for c in df_raw.columns if c.startswith(f"{emb_name}")],
        key=lambda x: int(x.split("_")[-1]),
    )
else:
    raise ValueError(f"File does not contain {model} columns.")

if not agdd_cols or not gdd_cols or not ndvi_cols:
    raise ValueError("Missing one or more daily baseline column groups (GDD, AGDD, NDVI).")

required_cols = (
    ["Year", "Latitude", "Longitude", "DOY"]
    + embedding_cols
    + agdd_cols
    + gdd_cols
    + ndvi_cols
)
df_clean = (
    df_raw.dropna(subset=required_cols)
    .sort_values(["Year", "DOY"])
    .reset_index(drop=True)
)

print(f"\nInitial records in file:        {len(df_raw)}")
print(f"Rows with complete {emb_name}:     {len(df_clean)}")
print(f"Years covered:                  {sorted(df_clean['Year'].unique())}")
print(f"{emb_name} channels detected:      {len(embedding_cols)}")

summary_save_path = (
    results_dir / f"{selected_folder.name}" / f"LOYOCV_{model}_C22_vs_DL"
)
summary_save_path.mkdir(parents=True, exist_ok=True)

### 4. Automated Time-Series Feature Extraction (`catch22` + Double Logistic Curves)

In [ ]:
print("Extracting catch22 features for GDD, AGDD, and NDVI...")

# 1. catch22 extraction for GDD (22 features)
gdd_matrix = df_clean[gdd_cols].values.astype(float)
gdd_c22_vals = np.array(
    [pycatch22.catch22_all(row)["values"] for row in tqdm(gdd_matrix, desc="GDD catch22")]
)
c22_names = pycatch22.catch22_all(gdd_matrix[0])["names"]
gdd_c22_cols = [f"gdd_c22_{n}" for n in c22_names]
df_gdd_c22 = pd.DataFrame(gdd_c22_vals, columns=gdd_c22_cols, index=df_clean.index)

# 2. catch22 extraction for AGDD (22 features)
agdd_matrix = df_clean[agdd_cols].values.astype(float)
agdd_c22_vals = np.array(
    [pycatch22.catch22_all(row)["values"] for row in tqdm(agdd_matrix, desc="AGDD catch22")]
)
agdd_c22_cols = [f"agdd_c22_{n}" for n in c22_names]
df_agdd_c22 = pd.DataFrame(agdd_c22_vals, columns=agdd_c22_cols, index=df_clean.index)

# 3. catch22 extraction for NDVI (22 features)
ndvi_matrix = df_clean[ndvi_cols].values.astype(float)
ndvi_c22_vals = np.array(
    [pycatch22.catch22_all(row)["values"] for row in tqdm(ndvi_matrix, desc="NDVI catch22")]
)
ndvi_c22_cols = [f"ndvi_c22_{n}" for n in c22_names]
df_ndvi_c22 = pd.DataFrame(ndvi_c22_vals, columns=ndvi_c22_cols, index=df_clean.index)

# Replace any inf values with NaN (imputed strictly inside each LOYOCV training fold)
for df_part in [df_gdd_c22, df_agdd_c22, df_ndvi_c22]:
    df_part.replace([np.inf, -np.inf], np.nan, inplace=True)

# 4. Double Logistic Curve Fitting on NDVI (7 biophysical parameters per station-year)
# Extracts: base_ndvi, amplitude, sos_doy, greenup_rate, eos_doy, senescence_rate, pos_doy
print("\nFitting Double Logistic curves to daily NDVI trajectories...")
t_days = np.arange(1, len(ndvi_cols) + 1, dtype=float)
dl_rows = []
dl_failures = 0

for row_ndvi in tqdm(ndvi_matrix, desc="Double Logistic NDVI"):
    w_base_init = float(np.percentile(row_ndvi, 10))
    w_amp_init = float(np.percentile(row_ndvi, 95) - w_base_init)
    w_amp_init = max(w_amp_init, 0.05)
    peak_day = float(np.argmax(row_ndvi) + 1)
    sos_init = max(40.0, min(peak_day - 35.0, 140.0))
    eos_init = max(peak_day + 35.0, min(peak_day + 70.0, 290.0))

    p0 = [w_base_init, w_amp_init, sos_init, 0.08, eos_init, 0.08]
    lower_bounds = [-0.2, 0.01, 15.0, 0.005, 110.0, 0.005]
    upper_bounds = [0.8, 1.2, 200.0, 0.5, 340.0, 0.5]

    try:
        popt, _ = curve_fit(
            lambda t, wb, wa, sos, r_up, eos, r_dn: wb
            + wa
            * (
                1.0 / (1.0 + np.exp(-np.clip(r_up * (t - sos), -50, 50)))
                - 1.0 / (1.0 + np.exp(-np.clip(r_dn * (t - eos), -50, 50)))
            ),
            t_days,
            row_ndvi,
            p0=p0,
            bounds=(lower_bounds, upper_bounds),
            maxfev=2500,
        )
        wb, wa, sos, r_up, eos, r_dn = popt
        fitted_curve = wb + wa * (
            1.0 / (1.0 + np.exp(-np.clip(r_up * (t_days - sos), -50, 50)))
            - 1.0 / (1.0 + np.exp(-np.clip(r_dn * (t_days - eos), -50, 50)))
        )
        pos_doy = float(np.argmax(fitted_curve) + 1)
        dl_rows.append([wb, wa, sos, r_up, eos, r_dn, pos_doy])
    except Exception:
        dl_failures += 1
        dl_rows.append([w_base_init, w_amp_init, sos_init, 0.08, eos_init, 0.08, peak_day])

ndvi_dl_cols = [
    "ndvi_dl_base",
    "ndvi_dl_amp",
    "ndvi_dl_sos_doy",
    "ndvi_dl_greenup_rate",
    "ndvi_dl_eos_doy",
    "ndvi_dl_senescence_rate",
    "ndvi_dl_pos_doy",
]
df_ndvi_dl = pd.DataFrame(dl_rows, columns=ndvi_dl_cols, index=df_clean.index)

# Combine all extracted features into df_features
df_features = pd.concat(
    [df_clean, df_gdd_c22, df_agdd_c22, df_ndvi_c22, df_ndvi_dl], axis=1
)
all_c22_cols = gdd_c22_cols + agdd_c22_cols + ndvi_c22_cols

print(f"Double Logistic fallback count: {dl_failures} / {len(df_clean)} rows")
print(f"Extracted feature counts -> GDD c22: {len(gdd_c22_cols)}, AGDD c22: {len(agdd_c22_cols)}, NDVI c22: {len(ndvi_c22_cols)}, NDVI DL: {len(ndvi_dl_cols)}")

### 5. Define Standalone Baselines (`B1`–`B5`) & FM Fusion Models (`M1`–`M6`)

In [ ]:
# Part A: Standalone Time-Series Baselines (No FM Embeddings)
baseline_feature_sets = {
    "B1_GDD_c22": gdd_c22_cols,
    "B2_AGDD_c22": agdd_c22_cols,
    "B3_NDVI_c22": ndvi_c22_cols,
    "B4_NDVI_DoubleLogistic": ndvi_dl_cols,
    "B5_All_c22_plus_DL": all_c22_cols + ndvi_dl_cols,
}

# Part B: Foundation Model + Time-Series Fusion Models
fusion_feature_sets = {
    "M1_FM_Only": embedding_cols,
    "M2_FM_plus_Spatial": ["Latitude", "Longitude"] + embedding_cols,
    "M3_FM_plus_GDD_c22": embedding_cols + gdd_c22_cols,
    "M4_FM_plus_AGDD_c22": embedding_cols + agdd_c22_cols,
    "M5a_FM_plus_NDVI_c22": embedding_cols + ndvi_c22_cols,
    "M5b_FM_plus_NDVI_DL": embedding_cols + ndvi_dl_cols,
    "M6_FM_plus_All_c22_DL": embedding_cols + all_c22_cols + ndvi_dl_cols,
}

all_model_sets = {**baseline_feature_sets, **fusion_feature_sets}

for k, cols in all_model_sets.items():
    print(f"  {k:26s} -> {len(cols):3d} features")

### 6. Run LOYOCV Hyperparameter Tuning & Evaluation (Random Forest & XGBoost)

In [ ]:
unique_years = sorted(df_features["Year"].unique())
y_all = df_features["DOY"].values
groups_all = df_features["Year"].values
logo = LeaveOneGroupOut()

rf_oof_preds = {}
xgb_oof_preds = {}
summary_records = []

print("=" * 95)
print(f"RUNNING LOYOCV BENCHMARK: STANDALONE BASELINES (B1-B5) vs. {model} FUSION (M1-M6)")
print("=" * 95)

for model_key, feat_cols in all_model_sets.items():
    print(f"\n--- {model_key} ({len(feat_cols)} features) ---")
    X_subset = df_features[feat_cols].copy()

    # Fill any NaNs using overall median only for RandomizedSearchCV; fold loop uses strict train-fold median
    X_subset_search = X_subset.fillna(X_subset.median()).fillna(0.0)

    max_feat_dist = [1.0] if len(feat_cols) <= 5 else uniform(0.25, 0.50)

    # 1. Tune Random Forest
    rf_param_dist = {
        "n_estimators": randint(150, 300),
        "max_depth": [12, 18, 25, None],
        "min_samples_leaf": randint(2, 6),
        "max_features": max_feat_dist,
    }
    rf_search = RandomizedSearchCV(
        estimator=RandomForestRegressor(random_state=42, n_jobs=1),
        param_distributions=rf_param_dist,
        n_iter=15,
        scoring="neg_mean_absolute_error",
        cv=logo,
        random_state=42,
        n_jobs=4,
    )
    rf_search.fit(X_subset_search, y_all, groups=groups_all)
    best_rf_params = rf_search.best_params_

    # 2. Tune XGBoost
    xgb_param_dist = {
        "n_estimators": randint(150, 350),
        "max_depth": randint(3, 7),
        "learning_rate": uniform(0.02, 0.08),
        "subsample": uniform(0.7, 0.3),
        "colsample_bytree": uniform(0.5, 0.5),
    }
    xgb_search = RandomizedSearchCV(
        estimator=XGBRegressor(random_state=42, n_jobs=1, verbosity=0),
        param_distributions=xgb_param_dist,
        n_iter=15,
        scoring="neg_mean_absolute_error",
        cv=logo,
        random_state=42,
        n_jobs=4,
    )
    xgb_search.fit(X_subset_search, y_all, groups=groups_all)
    best_xgb_params = xgb_search.best_params_

    # 3. Generate Out-of-Fold LOYOCV Predictions (strict training-fold median imputation)
    preds_rf = np.full(len(df_features), np.nan)
    preds_xgb = np.full(len(df_features), np.nan)

    for test_year in unique_years:
        train_mask = (df_features["Year"] != test_year).values
        test_mask = (df_features["Year"] == test_year).values

        X_train = df_features.loc[train_mask, feat_cols].copy()
        y_train = df_features.loc[train_mask, "DOY"].values
        X_test = df_features.loc[test_mask, feat_cols].copy()

        train_medians = X_train.median()
        X_train = X_train.fillna(train_medians).fillna(0.0)
        X_test = X_test.fillna(train_medians).fillna(0.0)

        rf_est = RandomForestRegressor(**best_rf_params, random_state=42, n_jobs=-1)
        rf_est.fit(X_train, y_train)
        preds_rf[test_mask] = rf_est.predict(X_test)

        xgb_est = XGBRegressor(**best_xgb_params, random_state=42, n_jobs=-1, verbosity=0)
        xgb_est.fit(X_train, y_train)
        preds_xgb[test_mask] = xgb_est.predict(X_test)

    rf_oof_preds[model_key] = preds_rf
    xgb_oof_preds[model_key] = preds_xgb

    # Compute Global Metrics
    rf_mae = mean_absolute_error(y_all, preds_rf)
    rf_rmse = np.sqrt(mean_squared_error(y_all, preds_rf))
    rf_r2 = r2_score(y_all, preds_rf)

    xgb_mae = mean_absolute_error(y_all, preds_xgb)
    xgb_rmse = np.sqrt(mean_squared_error(y_all, preds_xgb))
    xgb_r2 = r2_score(y_all, preds_xgb)

    summary_records.append({
        "Model_Key": model_key,
        "Group": "Baseline" if model_key.startswith("B") else f"{model}_Fusion",
        "N_Features": len(feat_cols),
        "RF_MAE": round(rf_mae, 2),
        "RF_RMSE": round(rf_rmse, 2),
        "RF_Global_R2": round(rf_r2, 3),
        "XGB_MAE": round(xgb_mae, 2),
        "XGB_RMSE": round(xgb_rmse, 2),
        "XGB_Global_R2": round(xgb_r2, 3),
    })

    print(
        f"  RF  -> MAE: {rf_mae:5.2f} d | RMSE: {rf_rmse:5.2f} d | R²: {rf_r2:6.3f}"
    )
    print(
        f"  XGB -> MAE: {xgb_mae:5.2f} d | RMSE: {xgb_rmse:5.2f} d | R²: {xgb_r2:6.3f}"
    )

df_benchmark = pd.DataFrame(summary_records)
display(df_benchmark)
df_benchmark.to_csv(
    summary_save_path / f"{model.lower()}_c22_vs_dl_benchmark.csv",
    sep=";",
    index=False,
)

### 7. Diagnostic Comparison Plots (Baselines vs. FM Fusion & $R^2$ Decomposition)

In [ ]:
actual_doy_arr = np.asarray(y_all)
years_arr = df_features["Year"].astype(int).values
station_ids = (
    df_features["PEP_ID"].values
    if "PEP_ID" in df_features.columns
    else (
        df_features["Latitude"].round(4).astype(str)
        + "_"
        + df_features["Longitude"].round(4).astype(str)
    ).values
)

all_keys = list(all_model_sets.keys())

# -------------------------------------------------------------------------
# Plot 1: R² Decomposition (Global vs. Spatial vs. Temporal Anomaly R²)
# -------------------------------------------------------------------------
fig, axes = plt.subplots(2, 1, figsize=(16, 11), sharex=True)

for ax, algo_name, preds_dict in [
    (axes[0], "Random Forest", rf_oof_preds),
    (axes[1], "XGBoost", xgb_oof_preds),
]:
    g_list, s_list, t_list = [], [], []
    for k in all_keys:
        preds = preds_dict[k]
        g_list.append(r2_score(actual_doy_arr, preds))

        df_tmp = pd.DataFrame(
            {"Station": station_ids, "Actual": actual_doy_arr, "Pred": preds}
        )
        st_means = df_tmp.groupby("Station")[["Actual", "Pred"]].mean()
        s_list.append(r2_score(st_means["Actual"], st_means["Pred"]))

        df_tmp = df_tmp.merge(st_means, on="Station", suffixes=("", "_mean"))
        t_list.append(
            r2_score(
                df_tmp["Actual"] - df_tmp["Actual_mean"],
                df_tmp["Pred"] - df_tmp["Pred_mean"],
            )
        )

    x_pos = np.arange(len(all_keys))
    w = 0.26
    ax.bar(x_pos - w, g_list, width=w, label="Global Pooled R²", color="steelblue")
    ax.bar(x_pos, s_list, width=w, label="Spatial Station-Mean R²", color="darkorange")
    ax.bar(x_pos + w, t_list, width=w, label="Temporal Anomaly R² (ΔDOY)", color="forestgreen")

    ax.axvline(4.5, color="gray", linestyle="--", lw=1.2)
    ax.axhline(0, color="black", lw=0.9)
    ax.set_ylabel("R² Score")
    ax.set_title(f"{algo_name}: Standalone Baselines (Left) vs. {model} Fusion (Right)")
    ax.grid(True, axis="y", linestyle=":", alpha=0.6)
    ax.legend(loc="upper left", frameon=True)

axes[1].set_xticks(np.arange(len(all_keys)))
axes[1].set_xticklabels(all_keys, rotation=25, ha="right")
plt.suptitle(
    f"catch22 vs. Double Logistic Phenology Curves — {model}\n{final_name}",
    fontsize=13,
)
plt.tight_layout()
plt.savefig(
    summary_save_path / f"r2_decomposition_{model.lower()}_c22_vs_dl.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

# -------------------------------------------------------------------------
# Plot 2: Direct Head-to-Head MAE Comparison (Baselines vs. FM Fusion)
# -------------------------------------------------------------------------
x_pos = np.arange(len(df_benchmark))
w = 0.35

plt.figure(figsize=(16, 6))
plt.bar(x_pos - w / 2, df_benchmark["RF_MAE"], width=w, label="Random Forest MAE", color="#2b5c8f")
plt.bar(x_pos + w / 2, df_benchmark["XGB_MAE"], width=w, label="XGBoost MAE", color="#e76f51")
plt.axvline(4.5, color="black", linestyle="--", lw=1.2, label="Baselines | FM Fusion")

plt.xticks(x_pos, df_benchmark["Model_Key"], rotation=25, ha="right")
plt.ylabel("LOYOCV Mean Absolute Error (Days)")
plt.title(
    f"LOYOCV MAE Comparison: catch22 & Double Logistic Baselines vs. {model} Fusion\n{final_name}"
)
plt.grid(True, axis="y", linestyle=":", alpha=0.6)
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig(
    summary_save_path / f"mae_comparison_{model.lower()}_c22_vs_dl.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()